# Aula 2 · Do texto ao modelo — passo a passo

Este notebook reproduz, com calma, o que fizemos na **primeira parte da Aula 2**:
pegar avaliações de e-commerce em português, **tratar o texto**, **treinar** um modelo
que adivinha se o cliente ficou satisfeito, **testar** no que ele nunca viu e **salvar** tudo.

> Você pode rodar célula por célula (Shift+Enter). Cada passo tem explicação.

**Pré-requisito:** ter feito o setup da Aula 1 (`pip install pandas duckdb scikit-learn pyarrow`).

## 1. Pegar os dados

Usamos as avaliações do Olist (mesmo dataset da Aula 1). Esta célula baixa um arquivo
pronto com o texto da avaliação e a nota.

In [ ]:
import urllib.request, pathlib
import pandas as pd

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("olist.parquet")
if not PATH.exists():
    print("Baixando..."); urllib.request.urlretrieve(URL, PATH)

df = pd.read_parquet(PATH, columns=["review_comment_message", "review_score"])
df = df.rename(columns={"review_comment_message": "review", "review_score": "nota"})
print(f"{len(df):,} avaliações")
df.head()

## 2. Criar o rótulo

O modelo precisa de uma "resposta certa" para cada exemplo. Definimos:
**nota 4 ou 5 = satisfeito (1)**, abaixo disso = insatisfeito (0).

In [ ]:
df["satisfeito"] = (df["nota"] >= 4).astype(int)
print(df["satisfeito"].value_counts())
print(f"\n{df['satisfeito'].mean()*100:.0f}% das avaliações são positivas")

## 3. Limpar o texto

Antes de treinar, padronizamos o texto. Três passos simples:
- **minúsculas** — para 'ÓTIMO' e 'ótimo' serem a mesma palavra
- **remover pontuação** — '!', '.', ',' não ajudam a decidir
- **vazios** — avaliações sem texto viram string vazia (senão o modelo quebra)

> Lembre da aula: **63% dos clientes não escrevem nada** — por isso o `fillna("")` é essencial.

In [ ]:
df["texto_limpo"] = (
    df["review"]
      .fillna("")                                  # sem texto -> vazio
      .str.lower()                                 # minúsculas
      .str.replace(r"[^a-zà-ú0-9 ]", "", regex=True)  # tira pontuação
)

# Veja o antes e o depois:
df[["review", "texto_limpo"]].dropna().head()

Vamos trabalhar só com as avaliações que **têm texto** (as outras seriam o caso do modelo tabular).

In [ ]:
com_texto = df[df["texto_limpo"].str.len() > 0].copy()
print(f"{len(com_texto):,} avaliações com texto ({len(com_texto)/len(df)*100:.0f}% do total)")

## 4. Salvar o texto tratado em Parquet

O dado limpo é um **artefato reutilizável** — salvamos para não precisar tratar de novo.
Como já está num DataFrame do pandas, salvar é uma linha:

In [ ]:
com_texto[["texto_limpo", "satisfeito"]].to_parquet("reviews_tratados.parquet")
print("Salvo em reviews_tratados.parquet")

## 5. Esconder uma parte para testar

Para saber se o modelo **aprendeu** (e não só decorou), escondemos 20% dos dados.
Treinamos nos 80% e testamos nos 20% escondidos.

> `random_state=42` faz a divisão ser sempre igual — assim o resultado é reproduzível.

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    com_texto["texto_limpo"], com_texto["satisfeito"],
    test_size=0.2, random_state=42, stratify=com_texto["satisfeito"]
)
print(f"Treino: {len(X_treino):,}  |  Teste (escondido): {len(X_teste):,}")

## 6. Treinar o modelo

Usamos um **Pipeline**: ele transforma o texto em números e treina o classificador,
tudo junto. Detalhe importante: o Pipeline só "aprende" com o treino — nunca espia o teste.

- `TfidfVectorizer` — transforma texto em números (conta palavras, dá mais peso às raras)
- `LogisticRegression` — aprende o peso de cada palavra para decidir 👍 ou 👎

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

modelo = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
modelo.fit(X_treino, y_treino)
print("Modelo treinado!")

## 7. Prever no que o modelo nunca viu

In [ ]:
from sklearn.metrics import accuracy_score

previsoes = modelo.predict(X_teste)
acc = accuracy_score(y_teste, previsoes)
print(f"Acurácia no teste: {acc*100:.1f}%")

### A nota sozinha engana — veja a matriz de confusão

Como a maioria é positiva, um modelo "preguiçoso" que chuta sempre positivo já acerta muito.
A matriz mostra os acertos e erros de cada lado.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_teste, previsoes)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=14)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["disse NEG", "disse POS"]); ax.set_yticklabels(["era NEG", "era POS"])
ax.set_title("Matriz de confusão")
plt.tight_layout(); plt.show()

## 8. Testar com frases suas

In [ ]:
for frase in [
    "Produto maravilhoso, chegou rápido!",
    "Veio quebrado e ninguém resolve",
    "demorou mas no fim deu certo",
]:
    p = modelo.predict_proba([frase])[0][1]
    print(f"{'👍' if p>=0.5 else '👎'} ({p:.0%})  {frase}")

## 9. Salvar as predições em Parquet

Fecha o ciclo: **coletar → tratar → treinar → prever → salvar**.

In [ ]:
resultado = pd.DataFrame({
    "texto": X_teste,
    "verdade": y_teste,
    "previsao": previsoes,
})
resultado.to_parquet("predicoes.parquet")
print(f"{len(resultado):,} predições salvas em predicoes.parquet")

## Recapitulando

Você acabou de fazer o **ciclo de vida do dado** de ponta a ponta:

| Etapa | O que fez |
|-------|-----------|
| Coletar | baixou as avaliações |
| Tratar | limpou o texto + salvou em Parquet |
| Treinar | ensinou um modelo a classificar |
| Prever | testou no que ele nunca viu |
| Salvar | guardou as predições em Parquet |

**Na Aula 4:** um agente de IA (LLM) vai fazer essa MESMA classificação **sem treinar nada** —
só lendo o texto. Será que ele bate este modelo? 🚀